In [14]:
!pip install groq -q

In [15]:
"""
====================================================
IMPORT LIBRARIES
====================================================
"""

import pandas as pd
import sqlite3

from groq import Groq

In [16]:
"""
====================================================
LOAD STUDENT DATASET
====================================================
"""

df = pd.read_csv("student_performance.csv")

print("Dataset Loaded Successfully")
print()

display(df.head())

Dataset Loaded Successfully



,student_id,name,age,gender,branch,attendance_pct,assignment_score,midterm_score,final_score,gpa,passed
0,1,Aarav Sharma,20,Male,CSE,85,78,72,76,7.6,Yes
1,2,Priya Patel,21,Female,ECE,92,88,85,89,8.9,Yes
2,3,Rohit Kumar,20,Male,MECH,67,55,60,58,5.8,Yes
3,4,Sneha Iyer,22,Female,CSE,95,92,90,94,9.4,Yes
4,5,Vikram Singh,21,Male,CIVIL,72,62,65,63,6.3,Yes


In [17]:
"""
====================================================
BASIC DATA EXPLORATION
====================================================
"""

print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

print("\nColumns\n")
print(df.columns)

print("\nDataset Info\n")
df.info()

print("\nStatistical Summary\n")
display(df.describe())

Rows : 30
Columns : 11

Columns

Index(['student_id', 'name', 'age', 'gender', 'branch', 'attendance_pct',
       'assignment_score', 'midterm_score', 'final_score', 'gpa', 'passed'],
      dtype='object')

Dataset Info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        30 non-null     int64  
 1   name              30 non-null     object 
 2   age               30 non-null     int64  
 3   gender            30 non-null     object 
 4   branch            30 non-null     object 
 5   attendance_pct    30 non-null     int64  
 6   assignment_score  30 non-null     int64  
 7   midterm_score     30 non-null     int64  
 8   final_score       30 non-null     int64  
 9   gpa               30 non-null     float64
 10  passed            30 non-null     object 
dtypes: float64(1), int64(6), object(4)
memory usage: 2.7+ KB

St

,student_id,age,attendance_pct,assignment_score,midterm_score,final_score,gpa
count,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000
mean,15.500000,20.933333,78.633333,71.566667,71.633333,72.933333,7.293333
std,8.803408,0.827682,14.006115,16.241036,13.634346,15.915582,1.591558
min,1.000000,20.000000,48.000000,38.000000,42.000000,40.000000,4.000000
25%,8.250000,20.000000,68.500000,59.000000,62.750000,60.750000,6.075000
50%,15.500000,21.000000,84.000000,76.500000,72.500000,76.500000,7.650000
75%,22.750000,22.000000,89.750000,85.500000,82.750000,86.500000,8.650000
max,30.000000,22.000000,96.000000,94.000000,92.000000,95.000000,9.500000


In [18]:
"""
====================================================
DATA CLEANING
====================================================
"""

# Remove duplicate rows

df = df.drop_duplicates()

# Remove missing values if any

df = df.dropna()

print("Data Cleaning Completed")
print("Remaining Rows :", len(df))

Data Cleaning Completed
Remaining Rows : 30


In [19]:
"""
====================================================
CREATE SQLITE DATABASE
====================================================
"""

conn = sqlite3.connect("students.db")

df.to_sql(
    "students",
    conn,
    if_exists="replace",
    index=False
)

print("Table Created Successfully")

Table Created Successfully


In [20]:
"""
====================================================
TEST SQL QUERIES
====================================================
"""

query = """
SELECT name,
       branch,
       final_score
FROM students
ORDER BY final_score DESC
LIMIT 5
"""

result = pd.read_sql(query, conn)

display(result)

,name,branch,final_score
0,Meera Krishnan,IT,95
1,Sneha Iyer,CSE,94
2,Lakshmi Chandran,CSE,92
3,Swathi Menon,IT,91
4,Priya Patel,ECE,89


In [21]:
"""
====================================================
INITIALIZE GROQ CLIENT
====================================================
"""

GROQ_API_KEY = "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"

client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq Connected")

Groq Connected


In [22]:
"""
====================================================
AI DATA ANALYST
====================================================

This function:

1. Reads dataset statistics
2. Reads user question
3. Sends both to Groq
4. Returns answer
====================================================
"""

def ask_ai(question):

    dataset_summary = f"""

Dataset Information

Total Students : {len(df)}

Columns:

{list(df.columns)}

Statistical Summary:

{df.describe().to_string()}

Sample Data:

{df.head(10).to_string()}

"""

    prompt = f"""

You are a Student Performance Data Analyst.

Analyze the dataset information carefully.

Answer only based on the dataset.

Dataset:

{dataset_summary}

Question:

{question}

"""

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],

        temperature=0.2
    )

    return response.choices[0].message.content

In [23]:
"""
====================================================
SQL + AI ANALYSIS
====================================================

User asks question

System:
1. Runs SQL
2. Sends result to AI
3. AI explains result
====================================================
"""

def sql_ai_analysis(sql_query, question):

    sql_result = pd.read_sql(sql_query, conn)

    prompt = f"""

You are a Student Performance Analyst.

Question:

{question}

SQL Result:

{sql_result.to_string()}

Explain the result in simple English.

Give meaningful insights.

"""

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    print("\nSQL Result\n")
    display(sql_result)

    print("\nAI Insight\n")

    print(response.choices[0].message.content)

In [24]:
sql_ai_analysis(

"""
SELECT name,
       branch,
       gpa
FROM students
ORDER BY gpa DESC
LIMIT 5
""",

"Who are the top students based on GPA?"

)


SQL Result



,name,branch,gpa
0,Meera Krishnan,IT,9.5
1,Sneha Iyer,CSE,9.4
2,Lakshmi Chandran,CSE,9.2
3,Swathi Menon,IT,9.1
4,Priya Patel,ECE,8.9



AI Insight

**Top Students Based on GPA:**

The top students with the highest GPA are:

1. Meera Krishnan from the IT branch with a GPA of 9.5
2. Sneha Iyer from the CSE branch with a GPA of 9.4
3. Lakshmi Chandran from the CSE branch with a GPA of 9.2

**Insights:**

* The top 3 students are from either the IT or CSE branches, indicating that these branches may have more students who excel academically.
* Meera Krishnan has the highest GPA, showing that she is an outstanding student in her field.
* The GPA range of the top students is between 9.1 and 9.5, which suggests that these students are consistently performing well in their studies.
* Priya Patel from the ECE branch has a GPA of 8.9, which is lower than the top 3 students but still a good score, indicating that she is a strong student in her field.

**Recommendations:**

* The university can consider providing additional support or resources to students in the ECE branch to help them improve their academic performance.
* Meera

In [25]:
sql_ai_analysis(

"""
SELECT branch,
       AVG(gpa) AS average_gpa
FROM students
GROUP BY branch
""",

"Which branch performs better academically?"

)


SQL Result



,branch,average_gpa
0,CIVIL,6.750000
1,CSE,7.420000
2,ECE,6.383333
3,IT,8.640000
4,MECH,7.220000



AI Insight

**Academic Performance Comparison**

After analyzing the data, it's clear that the **IT branch** performs the best academically, with an average GPA of 8.64. This is significantly higher than the other branches.

The ranking of branches by average GPA is:

1. IT (8.64)
2. CSE (7.42)
3. MECH (7.22)
4. CIVIL (6.75)
5. ECE (6.38)

**Insights:**

* The IT branch has a noticeable lead over the other branches, indicating that students in this branch tend to perform exceptionally well academically.
* The CSE branch comes in second, which is not surprising given the competitive nature of the field.
* The MECH branch performs relatively well, with an average GPA close to that of the CSE branch.
* The CIVIL and ECE branches have lower average GPAs, which may indicate areas where these branches could focus on improving academic support or resources for their students.

**Recommendations:**

* The IT branch can be used as a model to identify best practices that contribute to its stude

In [26]:
sql_ai_analysis(

"""
SELECT name,
       attendance_pct
FROM students
ORDER BY attendance_pct DESC
LIMIT 10
""",

"Who has the highest attendance?"
)


SQL Result



,name,attendance_pct
0,Meera Krishnan,96
1,Sneha Iyer,95
2,Lakshmi Chandran,94
3,Swathi Menon,93
4,Priya Patel,92
5,Divya Nair,91
6,Nandini Goswami,91
7,Kavitha Subramaniam,90
8,Rishika Shah,89
9,Ananya Reddy,88



AI Insight

**Attendance Leader: Meera Krishnan**

According to the data, Meera Krishnan has the highest attendance percentage, which is 96%. This means that Meera has been present in class 96% of the time, making her the most regular student.

**Key Insights:**

1. **Top Performers:** The top 3 students with the highest attendance are Meera Krishnan (96%), Sneha Iyer (95%), and Lakshmi Chandran (94%). These students have demonstrated excellent attendance habits and are likely to be more engaged in class.
2. **Average Attendance:** The average attendance percentage for the group is around 92%. This suggests that most students have good attendance habits, but there is still room for improvement.
3. **Areas for Improvement:** Students like Ananya Reddy (88%) and Rishika Shah (89%) may need to work on improving their attendance to catch up with their peers.

**Actionable Recommendations:**

1. **Recognize Meera's Achievement:** Consider recognizing Meera Krishnan's excellent attendance r

In [ ]:
"""
====================================================
AI DATA ANALYST CHATBOT
====================================================
"""

while True:

    question = input("\nAsk a Question : ")

    if question.lower() == "exit":
        break

    answer = ask_ai(question)

    print("\nAnswer:\n")
    print(answer)


Ask a Question : hi

Answer:

Hello. I'm ready to analyze the student performance dataset. What would you like to know about the data?

Ask a Question : who is the topper

Answer:

To determine the topper, we need to look at the 'gpa' column, which represents the student's overall grade point average. 

According to the dataset, the student with the highest GPA is Meera Krishnan (student_id: 10) with a GPA of 9.5, and Sneha Iyer (student_id: 4) also has a high GPA of 9.4. However, Meera Krishnan has the highest GPA among all students.

Therefore, the topper is Meera Krishnan.
